In [151]:
import numpy as np
import pandas as pd

print(pd.__version__)

2.2.3


In [4]:
import sys
print(sys.executable)

c:\Users\rauts\anaconda3\python.exe


In [119]:
df = pd.read_csv("emp_data.csv")
df

,employee_id,name,department,salary,joining_date,email,phone
0,101,John Doe,Engineering,85000,2021-05-12,john.doe@email.com,9.876421e+07
1,102,jane smith,HR,62000,2020/11/03,jane.smith@email.com,9.123458e+08
2,103,Mike Johnson,Engineering,NaN,2019-07-15,mike.j@email,9.988777e+09
3,104,Emily Davis,Finance,72000,15-08-2022,emily.davis@email.com,NaN
4,105,Chris Brown,Marketing,68000,2021-13-01,chris@email.com,8.877666e+09
5,106,Sarah Wilson,HR,not_available,2020-09-30,sarah.wilson@email.com,9.988341e+07
6,107,David Lee,Engineering,95000,2022-01-10,david.lee@email.com,9.876501e+09
7,108,NaN,Finance,71000,2021-06-18,finance@email.com,9.012346e+09
8,109,Anna Taylor,Marketing,67000,2021-04-25,anna.taylor@email.com,8.990011e+08
9,109,Anna Taylor,Marketing,67000,2021-04-25,anna.taylor@email.com,8.899001e+09


## Level 1 — Basic Cleaning

In [120]:
# 1  Remove extra spaces from names and salary fields
df['name'] = df['name'].str.strip().str.title()

# salary is object- removing extra space

df['salary'] = df['salary'].str.strip()

print(df['salary'].dtype)



object


In [121]:
# 2 Convert department names to consistent format
df['department'] = df['department'].str.strip().str.title()
df['department']

0     Engineering
1              Hr
2     Engineering
3         Finance
4       Marketing
5              Hr
6     Engineering
7         Finance
8       Marketing
9       Marketing
10    Engineering
11        Finance
12    Engineering
13             Hr
14      Marketing
15        Finance
Name: department, dtype: object

In [122]:
# 3 Fill missing names with "Unknown" -- Name column
df['name'] = df['name'].fillna("Unknown")
df['name']

0         John Doe
1       Jane Smith
2     Mike Johnson
3      Emily Davis
4      Chris Brown
5     Sarah Wilson
6        David Lee
7          Unknown
8      Anna Taylor
9      Anna Taylor
10     Robert King
11      Lisa White
12      Tom Harris
13      Nina Patel
14     Kevin Scott
15    Olivia Green
Name: name, dtype: object

In [123]:
# 4 Replace missing phone numbers with "Not Available"
# as phone column has datatype as "float" 
# convert the datatype to int and add 0 where num is missing


df['phone'] = df['phone'].fillna(0).astype(int)

df.dtypes

# after this phone col is int64

employee_id      int64
name            object
department      object
salary          object
joining_date    object
email           object
phone            int64
dtype: object

## Level 2 — Data Validation

In [76]:
# Find invalid emails
list_of_email = df['email'].tolist()
print(list_of_email)
invalid_email = []


['john.doe@email.com', 'jane.smith@email.com', 'mike.j@email', 'emily.davis@email.com', 'chris@email.com', 'sarah.wilson@email.com', 'david.lee@email.com', 'finance@email.com', 'anna.taylor@email.com', 'anna.taylor@email.com', 'robert.king@email.com', 'lisa.white@email.com', 'tom.harris@email.com', 'nina.patel@email', 'kevin.scott@email.com', 'olivia.green@email.com']


In [124]:
# Find invalid phone numbers

list_of_phone = df['phone'].tolist()

invalid_num = []

print("List of phones ", list_of_phone)
for phone in list_of_phone:
    phone_str = str(phone)
    #print(type(phone_str))

    if(len(phone_str) < 10):
        invalid_num.append(phone_str)

print("invalid nums ",invalid_num)



List of phones  [98764210, 912345780, 9988776655, 0, 8877665544, 99883412, 9876501234, 9012345678, 899001122, 8899001122, 12345, 9090909090, 998871122, 8877002211, 0, 9098765432]
invalid nums  ['98764210', '912345780', '0', '99883412', '899001122', '12345', '998871122', '0']


In [125]:
# Convert salary to integer and Handle "not_available" salary values and Handle negative salary values
# as salary is object with nulls, negative and string first treat the null then convert the salary to int
df['salary'] = df['salary'].replace(['not_available','NA','Invalid'], np.nan) # replace invalid strings with nan(float type)

# Convert object salary col to float
df['salary'] = df['salary'].astype(float)

# fill nan with 0
df['salary'] = df['salary'].fillna(0).astype(int)

# converting salary col to absolute values, as negative salary is not accepted
df['salary'] = df['salary'].abs()
print("Salary col is now",df['salary'].dtype)


Salary col is now int64


# Level 3 — Date Cleaning

In [ ]:
print(df['joining_date'].dtype)

# problem - join date is object with ugly format : eg: 2025-09-12, 2026/09/12 - put into 1 format
# Handle invalid dates like:2021-13-01


df['joining_date']

object


0     2021-05-12
1     2020/11/03
2     2019-07-15
3     15-08-2022
4     2021-13-01
5     2020-09-30
6     2022-01-10
7     2021-06-18
8     2021-04-25
9     2021-04-25
10    2023-02-01
11           NaN
12    2022-10-11
13    2021-08-20
14    2020-05-17
15    2022-12-12
Name: joining_date, dtype: object

In [ ]:
# Standardize all dates into: YYYY-MM-DD
# By-default pandas assumes Dates in format of MM-DD-YY - US format

### Still confusing

# Normalize separators
df['joining_date'] = df['joining_date'].str.replace('/', '-', regex=False)

# intially dayfirst as false, to handel yyyy-mm-dd or yyyy/mm/dd properly
clean_up1 = pd.to_datetime(df['joining_date'], format='mixed', dayfirst=False, errors='coerce')

# take rows which are NaT
nat_rows = clean_up1.isna()

# intially dayfirst as true on nat rows from clean_up1 
clean_up2 = pd.to_datetime(df['joining_date'][nat_rows], dayfirst=True, errors='coerce')  # this line only picks nat rows

clean_up1[nat_rows] = clean_up2.values

# Convert to final format - YYYY-MM-DD
df['joining_date'] = clean_up1.dt.strftime('%Y-%m-%d')
df['joining_date']

0     2021-05-12
1            NaN
2     2019-07-15
3            NaN
4            NaN
5     2020-09-30
6     2022-01-10
7     2021-06-18
8     2021-04-25
9     2021-04-25
10    2023-02-01
11           NaN
12    2022-10-11
13    2021-08-20
14    2020-05-17
15    2022-12-12
Name: joining_date, dtype: object

employee_id     0
name            0
department      0
salary          0
joining_date    4
email           0
phone           0
dtype: int64

In [ ]:
# Remove duplicate employee records -- Anna Taylor has repeated twice, with same emp_id, where the phone number is different
# as emp_id should be unique, we can drop 1 occurance of row as phone number wont matter much

df.drop_duplicates(subset=['employee_id','name', 'department', 'joining_date'], keep='first', inplace=True)

df


,employee_id,name,department,salary,joining_date,email,phone
0,101,John Doe,Engineering,85000,2021-05-12,john.doe@email.com,98764210
1,102,Jane Smith,Hr,62000,NaN,jane.smith@email.com,912345780
2,103,Mike Johnson,Engineering,0,2019-07-15,mike.j@email,9988776655
3,104,Emily Davis,Finance,72000,NaN,emily.davis@email.com,0
4,105,Chris Brown,Marketing,68000,NaN,chris@email.com,8877665544
5,106,Sarah Wilson,Hr,0,2020-09-30,sarah.wilson@email.com,99883412
6,107,David Lee,Engineering,95000,2022-01-10,david.lee@email.com,9876501234
7,108,Unknown,Finance,71000,2021-06-18,finance@email.com,9012345678
8,109,Anna Taylor,Marketing,67000,2021-04-25,anna.taylor@email.com,899001122
10,110,Robert King,Engineering,120000,2023-02-01,robert.king@email.com,12345


In [157]:
df.dtypes

employee_id      int64
name            object
department      object
salary           int64
joining_date    object
email           object
phone            int64
dtype: object

In [158]:
# Save the clean data in new csv file

df_cleaned = df
df_cleaned.to_csv('emp_clean_data.csv', index=False)